# Install dependencies

In [1]:
import sys
import subprocess
import importlib.util

packages = {
    "PIL": "pillow",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "imgutils": "dghs-imgutils",
    "huggingface_hub": "hugginface_hub",
    "ipywidgets": "ipywidgets"
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependency check completed.")

Dependency check completed.


# Imports

In [2]:
from pathlib import Path
from PIL import Image, ImageOps, UnidentifiedImageError
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json
import shutil
import os
import sys
import platform
import warnings
import traceback
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

Image.MAX_IMAGE_PIXELS = 300_000_000
warnings.simplefilter("error", Image.DecompressionBombWarning)

try:
    from imgutils.detect import detect_heads, detect_person
except Exception as exc:
    raise RuntimeError(
        "Could not import detect_heads and detect_person from dghs-imgutils. "
        "Restart the kernel after installation and run again."
    ) from exc

print("Imports completed.")

Imports completed.


# Configuration

In [3]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / f"02_crop_preparation_{RUN_ID}"

REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = RUN_DIR / "tables"
CROPS_DIR = RUN_DIR / "crops"
CONTACT_SHEETS_DIR = RUN_DIR / "contact_sheets"
LOGS_DIR = RUN_DIR / "logs"

USE_PERCEPTUAL_REPRESENTATIVES = False
MAX_WORKERS = 1

HEAD_CONF_THRESHOLD = 0.40
HEAD_IOU_THRESHOLD = 0.70

PERSON_CONF_THRESHOLD = 0.30
PERSON_IOU_THRESHOLD = 0.50

MIN_CROP_SIDE = 32
HEAD_PADDING_RATIO = 0.55
PERSON_PADDING_RATIO = 0.08

CROP_OUTPUT_SIZE = None
CROP_FORMAT = "JPEG"
CROP_QUALITY = 95

MAX_CONTACT_SHEET_GROUPS = 30
CONTACT_SHEET_THUMB_SIZE = 180
CONTACT_SHEET_COLUMNS = 5
CONTACT_SHEET_MAX_IMAGES_PER_GROUP = 20

for directory in [RUN_DIR, REPORTS_DIR, TABLES_DIR, CROPS_DIR, CONTACT_SHEETS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")


Project directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline
Run directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/02_crop_preparation_20260521_180111


# Locate previous audit run and load tables

In [4]:
def find_latest_audit_run(project_dir):
    runs_dir = Path(project_dir) / "runs"
    candidates = sorted(
        [p for p in runs_dir.glob("01_dataset_audit_*") if p.is_dir()],
        key=lambda p: p.name,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No previous audit run was found in {runs_dir}. "
            "Run notebook 01 before this notebook."
        )
    return candidates[-1]


PREVIOUS_RUN_DIR = globals().get("PREVIOUS_RUN_DIR", None)

if PREVIOUS_RUN_DIR is None:
    PREVIOUS_RUN_DIR = find_latest_audit_run(PROJECT_DIR)
else:
    PREVIOUS_RUN_DIR = Path(PREVIOUS_RUN_DIR).expanduser().resolve()

previous_tables_dir = PREVIOUS_RUN_DIR / "tables"

valid_images_path = previous_tables_dir / "valid_images.csv"
exact_duplicate_groups_path = previous_tables_dir / "exact_duplicate_groups.csv"
perceptual_duplicate_groups_path = previous_tables_dir / "perceptual_duplicate_groups.csv"
invalid_images_path = previous_tables_dir / "invalid_images.csv"

required_paths = [valid_images_path]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

valid_df = pd.read_csv(valid_images_path)

if exact_duplicate_groups_path.exists():
    exact_duplicates_df = pd.read_csv(exact_duplicate_groups_path)
else:
    exact_duplicates_df = pd.DataFrame()

if perceptual_duplicate_groups_path.exists():
    perceptual_groups_df = pd.read_csv(perceptual_duplicate_groups_path)
else:
    perceptual_groups_df = pd.DataFrame()

if invalid_images_path.exists():
    invalid_df = pd.read_csv(invalid_images_path)
else:
    invalid_df = pd.DataFrame()

print(f"Previous audit run: {PREVIOUS_RUN_DIR}")
print(f"Valid images loaded: {len(valid_df)}")
print(f"Exact duplicate rows loaded: {len(exact_duplicates_df)}")
print(f"Perceptual duplicate rows loaded: {len(perceptual_groups_df)}")
print(f"Invalid rows loaded: {len(invalid_df)}")

Previous audit run: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744
Valid images loaded: 15742
Exact duplicate rows loaded: 39
Perceptual duplicate rows loaded: 80
Invalid rows loaded: 2


# Helper functions

In [5]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def safe_int(value, default=None):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def safe_float(value, default=None):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def normalize_path_string(value):
    return Path(str(value)).expanduser().resolve().as_posix()


def clamp_box(box, width, height):
    x0, y0, x1, y1 = box
    x0 = max(0, min(int(round(x0)), width))
    y0 = max(0, min(int(round(y0)), height))
    x1 = max(0, min(int(round(x1)), width))
    y1 = max(0, min(int(round(y1)), height))
    if x1 < x0:
        x0, x1 = x1, x0
    if y1 < y0:
        y0, y1 = y1, y0
    return x0, y0, x1, y1


def expand_box(box, width, height, ratio):
    x0, y0, x1, y1 = box
    bw = max(1, x1 - x0)
    bh = max(1, y1 - y0)
    px = bw * ratio
    py = bh * ratio
    return clamp_box((x0 - px, y0 - py, x1 + px, y1 + py), width, height)


def box_area(box):
    x0, y0, x1, y1 = box
    return max(0, x1 - x0) * max(0, y1 - y0)


def detection_to_dict(det):
    box, label, score = det
    x0, y0, x1, y1 = box
    return {
        "x0": int(x0),
        "y0": int(y0),
        "x1": int(x1),
        "y1": int(y1),
        "label": str(label),
        "score": float(score),
    }


def choose_best_detection(detections, image_width, image_height):
    if not detections:
        return None

    image_area = max(1, image_width * image_height)

    def score_detection(det):
        box, label, score = det
        area = box_area(clamp_box(box, image_width, image_height))
        area_ratio = area / image_area
        return float(score) + 0.10 * min(area_ratio, 1.0)

    return max(detections, key=score_detection)


def make_crop_id(index_value, path_value):
    stem = Path(str(path_value)).stem
    safe_stem = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in stem)
    safe_stem = safe_stem[:80].strip("_") or "image"
    return f"img_{int(index_value):07d}_{safe_stem}"


def save_rgb_image(image, output_path, crop_format, quality):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if crop_format.upper() == "JPEG":
        image.convert("RGB").save(output_path, format="JPEG", quality=quality, optimize=True)
    elif crop_format.upper() == "PNG":
        image.save(output_path, format="PNG", optimize=True)
    else:
        raise ValueError(f"Unsupported CROP_FORMAT: {crop_format}")


def create_contact_sheet(image_paths, output_path, thumb_size=180, columns=5):
    image_paths = [Path(p) for p in image_paths if Path(p).exists()]
    if not image_paths:
        return False

    rows = int(np.ceil(len(image_paths) / columns))
    sheet_width = columns * thumb_size
    sheet_height = rows * thumb_size
    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")

    for idx, path in enumerate(image_paths):
        try:
            with Image.open(path) as img:
                img = ImageOps.exif_transpose(img.convert("RGB"))
                img.thumbnail((thumb_size, thumb_size), Image.Resampling.LANCZOS)
                x = (idx % columns) * thumb_size + (thumb_size - img.width) // 2
                y = (idx // columns) * thumb_size + (thumb_size - img.height) // 2
                sheet.paste(img, (x, y))
        except Exception:
            continue

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, format="JPEG", quality=92, optimize=True)
    return True


print("Helper functions loaded.")

Helper functions loaded.


# Construct clean representative image set

In [6]:
working_df = valid_df.copy()
working_df["path"] = working_df["path"].map(normalize_path_string)

excluded_exact_paths = set()

if not exact_duplicates_df.empty:
    exact_df = exact_duplicates_df.copy()
    exact_df["path"] = exact_df["path"].map(normalize_path_string)
    non_rep_exact_df = exact_df[~exact_df["is_representative"].astype(bool)].copy()
    excluded_exact_paths = set(non_rep_exact_df["path"].tolist())

excluded_perceptual_paths = set()

if USE_PERCEPTUAL_REPRESENTATIVES and not perceptual_groups_df.empty:
    perceptual_df = perceptual_groups_df.copy()
    perceptual_df["path"] = perceptual_df["path"].map(normalize_path_string)
    non_rep_perceptual_df = perceptual_df[~perceptual_df["is_representative"].astype(bool)].copy()
    excluded_perceptual_paths = set(non_rep_perceptual_df["path"].tolist())

working_df["excluded_exact_duplicate"] = working_df["path"].isin(excluded_exact_paths)
working_df["excluded_perceptual_duplicate"] = working_df["path"].isin(excluded_perceptual_paths)
working_df["selected_for_detection"] = ~working_df["excluded_exact_duplicate"] & ~working_df["excluded_perceptual_duplicate"]

selected_df = working_df[working_df["selected_for_detection"]].copy().reset_index(drop=True)
selected_df.insert(0, "image_index", np.arange(len(selected_df), dtype=int))

selection_path = TABLES_DIR / "selected_images_for_detection.csv"
working_path = TABLES_DIR / "valid_images_with_selection_flags.csv"

selected_df.to_csv(selection_path, index=False)
working_df.to_csv(working_path, index=False)

print(f"Valid images: {len(working_df)}")
print(f"Excluded exact duplicate non-representatives: {int(working_df['excluded_exact_duplicate'].sum())}")
print(f"Excluded perceptual duplicate non-representatives: {int(working_df['excluded_perceptual_duplicate'].sum())}")
print(f"Selected for detection: {len(selected_df)}")

if selected_df.empty:
    raise RuntimeError("No images were selected for detection.")

Valid images: 15742
Excluded exact duplicate non-representatives: 37
Excluded perceptual duplicate non-representatives: 0
Selected for detection: 15705


# Detection and crop extraction function

In [7]:
def process_image_for_crop(row):
    path = Path(row["path"])
    image_index = int(row["image_index"])
    crop_id = make_crop_id(image_index, path)
    crop_extension = ".jpg" if CROP_FORMAT.upper() == "JPEG" else ".png"
    crop_path = CROPS_DIR / f"{crop_id}{crop_extension}"

    record = {
        "image_index": image_index,
        "crop_id": crop_id,
        "source_path": path.as_posix(),
        "relative_path": row.get("relative_path"),
        "crop_path": crop_path.as_posix(),
        "status": "unknown",
        "error_type": None,
        "error_message": None,
        "source_width": None,
        "source_height": None,
        "crop_width": None,
        "crop_height": None,
        "selected_region_type": None,
        "selected_score": None,
        "detected_heads_count": 0,
        "detected_persons_count": 0,
        "is_multi_head": False,
        "is_multi_person": False,
        "selected_x0": None,
        "selected_y0": None,
        "selected_x1": None,
        "selected_y1": None,
        "head_detections_json": None,
        "person_detections_json": None,
        "processed_at": now_iso(),
    }

    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img.convert("RGB"))
            width, height = img.size

            record["source_width"] = int(width)
            record["source_height"] = int(height)

            try:
                head_detections = detect_heads(
                    img,
                    conf_threshold=HEAD_CONF_THRESHOLD,
                    iou_threshold=HEAD_IOU_THRESHOLD,
                )
            except Exception:
                head_detections = []

            try:
                person_detections = detect_person(
                    img,
                    conf_threshold=PERSON_CONF_THRESHOLD,
                    iou_threshold=PERSON_IOU_THRESHOLD,
                )
            except Exception:
                person_detections = []

            head_detections = [
                (clamp_box(det[0], width, height), det[1], float(det[2]))
                for det in head_detections
                if box_area(clamp_box(det[0], width, height)) > 0
            ]

            person_detections = [
                (clamp_box(det[0], width, height), det[1], float(det[2]))
                for det in person_detections
                if box_area(clamp_box(det[0], width, height)) > 0
            ]

            record["detected_heads_count"] = int(len(head_detections))
            record["detected_persons_count"] = int(len(person_detections))
            record["is_multi_head"] = bool(len(head_detections) > 1)
            record["is_multi_person"] = bool(len(person_detections) > 1)
            record["head_detections_json"] = json.dumps([detection_to_dict(det) for det in head_detections], ensure_ascii=False)
            record["person_detections_json"] = json.dumps([detection_to_dict(det) for det in person_detections], ensure_ascii=False)

            best_head = choose_best_detection(head_detections, width, height)
            best_person = choose_best_detection(person_detections, width, height)

            if best_head is not None:
                selected_box = expand_box(best_head[0], width, height, HEAD_PADDING_RATIO)
                selected_type = "head"
                selected_score = float(best_head[2])
            elif best_person is not None:
                selected_box = expand_box(best_person[0], width, height, PERSON_PADDING_RATIO)
                selected_type = "person"
                selected_score = float(best_person[2])
            else:
                selected_box = (0, 0, width, height)
                selected_type = "full_image"
                selected_score = None

            x0, y0, x1, y1 = clamp_box(selected_box, width, height)

            if (x1 - x0) < MIN_CROP_SIDE or (y1 - y0) < MIN_CROP_SIDE:
                x0, y0, x1, y1 = 0, 0, width, height
                selected_type = "full_image_small_detection_fallback"
                selected_score = None

            crop = img.crop((x0, y0, x1, y1))

            if CROP_OUTPUT_SIZE is not None:
                crop = crop.resize((int(CROP_OUTPUT_SIZE), int(CROP_OUTPUT_SIZE)), Image.Resampling.LANCZOS)

            save_rgb_image(crop, crop_path, CROP_FORMAT, CROP_QUALITY)

            record["status"] = "ok"
            record["crop_width"] = int(crop.width)
            record["crop_height"] = int(crop.height)
            record["selected_region_type"] = selected_type
            record["selected_score"] = selected_score
            record["selected_x0"] = int(x0)
            record["selected_y0"] = int(y0)
            record["selected_x1"] = int(x1)
            record["selected_y1"] = int(y1)

            return record

    except Exception as exc:
        record["status"] = "error"
        record["error_type"] = type(exc).__name__
        record["error_message"] = str(exc)
        return record


# Run detection and crop extraction

In [ ]:
crop_records = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_image_for_crop, row): int(row["image_index"])
        for _, row in selected_df.iterrows()
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Detecting regions and creating crops"):
        try:
            crop_records.append(future.result())
        except Exception as exc:
            crop_records.append({
                "image_index": futures[future],
                "crop_id": None,
                "source_path": None,
                "relative_path": None,
                "crop_path": None,
                "status": "error",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "source_width": None,
                "source_height": None,
                "crop_width": None,
                "crop_height": None,
                "selected_region_type": None,
                "selected_score": None,
                "detected_heads_count": 0,
                "detected_persons_count": 0,
                "is_multi_head": False,
                "is_multi_person": False,
                "selected_x0": None,
                "selected_y0": None,
                "selected_x1": None,
                "selected_y1": None,
                "head_detections_json": None,
                "person_detections_json": None,
                "processed_at": now_iso(),
            })

crops_df = pd.DataFrame(crop_records).sort_values("image_index").reset_index(drop=True)

print(f"Processed images: {len(crops_df)}")
print(f"Successful crops: {int(crops_df['status'].eq('ok').sum())}")
print(f"Crop errors: {int(crops_df['status'].ne('ok').sum())}")

Detecting regions and creating crops:   0%|          | 0/15705 [00:00<?, ?it/s]

# Classify crop quality and review categories

In [ ]:
review_df = crops_df.copy()

review_df["needs_review"] = False
review_df["review_reason"] = ""

review_df.loc[review_df["status"].ne("ok"), "needs_review"] = True
review_df.loc[review_df["status"].ne("ok"), "review_reason"] = "crop_error"

review_df.loc[
    review_df["selected_region_type"].eq("full_image"),
    "review_reason"
] = review_df.loc[
    review_df["selected_region_type"].eq("full_image"),
    "review_reason"
].replace("", "no_detection_full_image_fallback")

review_df.loc[
    review_df["selected_region_type"].eq("full_image"),
    "needs_review"
] = True

review_df.loc[
    review_df["selected_region_type"].eq("full_image_small_detection_fallback"),
    "needs_review"
] = True

review_df.loc[
    review_df["selected_region_type"].eq("full_image_small_detection_fallback"),
    "review_reason"
] = "small_detection_full_image_fallback"

review_df.loc[
    review_df["is_multi_head"].astype(bool),
    "needs_review"
] = True

review_df.loc[
    review_df["is_multi_head"].astype(bool) & review_df["review_reason"].eq(""),
    "review_reason"
] = "multiple_heads_detected"

review_df.loc[
    review_df["is_multi_head"].astype(bool) & ~review_df["review_reason"].eq("multiple_heads_detected"),
    "review_reason"
] = review_df.loc[
    review_df["is_multi_head"].astype(bool) & ~review_df["review_reason"].eq("multiple_heads_detected"),
    "review_reason"
] + ";multiple_heads_detected"

region_counts = review_df["selected_region_type"].value_counts(dropna=False).reset_index()
region_counts.columns = ["selected_region_type", "count"]

review_counts = review_df["review_reason"].value_counts(dropna=False).reset_index()
review_counts.columns = ["review_reason", "count"]

print("Selected region types:")
display(region_counts)

print("Review reasons:")
display(review_counts)

# Save crop manifest and reports

In [ ]:
crops_manifest_path = TABLES_DIR / "crop_manifest.csv"
review_manifest_path = TABLES_DIR / "crop_manifest_with_review_flags.csv"
region_counts_path = TABLES_DIR / "selected_region_counts.csv"
review_counts_path = TABLES_DIR / "review_reason_counts.csv"

crops_df.to_csv(crops_manifest_path, index=False)
review_df.to_csv(review_manifest_path, index=False)
region_counts.to_csv(region_counts_path, index=False)
review_counts.to_csv(review_counts_path, index=False)

summary = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "project_dir": PROJECT_DIR.as_posix(),
    "previous_run_dir": PREVIOUS_RUN_DIR.as_posix(),
    "run_dir": RUN_DIR.as_posix(),
    "use_perceptual_representatives": bool(USE_PERCEPTUAL_REPRESENTATIVES),
    "valid_images_loaded": int(len(valid_df)),
    "selected_for_detection": int(len(selected_df)),
    "successful_crops": int(crops_df["status"].eq("ok").sum()),
    "crop_errors": int(crops_df["status"].ne("ok").sum()),
    "head_conf_threshold": float(HEAD_CONF_THRESHOLD),
    "head_iou_threshold": float(HEAD_IOU_THRESHOLD),
    "person_conf_threshold": float(PERSON_CONF_THRESHOLD),
    "person_iou_threshold": float(PERSON_IOU_THRESHOLD),
    "head_padding_ratio": float(HEAD_PADDING_RATIO),
    "person_padding_ratio": float(PERSON_PADDING_RATIO),
    "crop_format": CROP_FORMAT,
    "crop_quality": int(CROP_QUALITY),
    "crop_output_size": CROP_OUTPUT_SIZE,
    "python": sys.version,
    "platform": platform.platform(),
    "outputs": {
        "selected_images_for_detection": selection_path.as_posix(),
        "valid_images_with_selection_flags": working_path.as_posix(),
        "crop_manifest": crops_manifest_path.as_posix(),
        "crop_manifest_with_review_flags": review_manifest_path.as_posix(),
        "selected_region_counts": region_counts_path.as_posix(),
        "review_reason_counts": review_counts_path.as_posix(),
        "crops_dir": CROPS_DIR.as_posix(),
    },
}

summary_path = REPORTS_DIR / "summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved outputs:")
for key, value in summary["outputs"].items():
    print(f"{key}: {value}")

print(f"summary: {summary_path.as_posix()}")

# Create representative contact sheets

In [ ]:
contact_sheet_rows = []

ok_review_df = review_df[review_df["status"].eq("ok")].copy()

groups_for_sheets = []

for region_type, group in ok_review_df.groupby("selected_region_type", dropna=False):
    groups_for_sheets.append((f"region_{region_type}", group))

needs_review_group = ok_review_df[ok_review_df["needs_review"].astype(bool)].copy()
if not needs_review_group.empty:
    groups_for_sheets.append(("needs_review", needs_review_group))

multi_head_group = ok_review_df[ok_review_df["is_multi_head"].astype(bool)].copy()
if not multi_head_group.empty:
    groups_for_sheets.append(("multiple_heads_detected", multi_head_group))

for sheet_name, group in groups_for_sheets[:MAX_CONTACT_SHEET_GROUPS]:
    group = group.head(CONTACT_SHEET_MAX_IMAGES_PER_GROUP)
    image_paths = group["crop_path"].dropna().tolist()
    output_path = CONTACT_SHEETS_DIR / f"{sheet_name}.jpg"
    created = create_contact_sheet(
        image_paths=image_paths,
        output_path=output_path,
        thumb_size=CONTACT_SHEET_THUMB_SIZE,
        columns=CONTACT_SHEET_COLUMNS,
    )
    contact_sheet_rows.append({
        "sheet_name": sheet_name,
        "image_count": len(image_paths),
        "created": bool(created),
        "path": output_path.as_posix() if created else None,
    })

contact_sheets_df = pd.DataFrame(contact_sheet_rows)
contact_sheets_path = TABLES_DIR / "contact_sheets.csv"
contact_sheets_df.to_csv(contact_sheets_path, index=False)

contact_sheets_df

# Inspect final manifest

In [ ]:
review_df[[
    "image_index",
    "relative_path",
    "status",
    "selected_region_type",
    "selected_score",
    "detected_heads_count",
    "detected_persons_count",
    "is_multi_head",
    "needs_review",
    "review_reason",
    "crop_path",
]].head(30)

# Inspect possible problems

In [ ]:
problem_df = review_df[
    review_df["status"].ne("ok") |
    review_df["needs_review"].astype(bool)
].copy()

if problem_df.empty:
    print("No crop errors or review cases were detected.")
else:
    problem_df[[
        "image_index",
        "relative_path",
        "status",
        "error_type",
        "error_message",
        "selected_region_type",
        "detected_heads_count",
        "detected_persons_count",
        "review_reason",
    ]].head(80)

# Verify crop files exist

In [ ]:
existing_crop_count = review_df["crop_path"].dropna().map(lambda p: Path(p).exists()).sum()
expected_crop_count = int(review_df["status"].eq("ok").sum())

verification = {
    "expected_successful_crops": expected_crop_count,
    "existing_crop_files": int(existing_crop_count),
    "all_successful_crops_exist": bool(existing_crop_count == expected_crop_count),
}

verification_path = REPORTS_DIR / "crop_file_verification.json"

with verification_path.open("w", encoding="utf-8") as f:
    json.dump(verification, f, ensure_ascii=False, indent=2)

verification